In [1]:
'''Using option prices to estimate an implied-volatility surface.
Applying the Dupire formula to convert it into a local-volatility surface.
Using that local volatility in Black-Scholes to calculate the option’s delta.
For one short call, buying approximately delta × 100 SPY shares.
Rebalancing the shares daily as the calculated delta changes.'''

'Using option prices to estimate an implied-volatility surface.\nApplying the Dupire formula to convert it into a local-volatility surface.\nUsing that local volatility in Black-Scholes to calculate the option’s delta.\nFor one short call, buying approximately delta × 100 SPY shares.\nRebalancing the shares daily as the calculated delta changes.'

In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import LinearNDInterpolator
#1. Using option prices to estimate an implied-volatility surface.

options = pd.read_csv("../../data/surface_options.csv")

options["date"] = pd.to_datetime(options["date"])
options["expiration"] = pd.to_datetime(options["expiration"])

# Calculate time to expiration in years
options["T"] = (
    options["expiration"] - options["date"]
).dt.days / 365

# Calculate strike relative to the stock price
options["log_moneyness"] = np.log(
    options["strike"] / options["underlying_price"]
)

# Build the implied-volatility surface
iv_surface = LinearNDInterpolator(
    options[["log_moneyness", "T"]],
    options["impliedVolatility"],
)

# Estimate IV for an ATM option expiring in 30 days
estimated_iv = iv_surface(0.0, 30 / 365)
print("Estimated implied volatility:", estimated_iv)

Estimated implied volatility: 0.1914676435816199


In [3]:
from scipy.stats import norm
import numpy as np

#2. Applying the Dupire formula to convert it into a local-volatility surface.
r = 0.04
spot = options["underlying_price"].median()

# Black-Scholes call price
def call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (
        sigma * np.sqrt(T)
    )
    d2 = d1 - sigma * np.sqrt(T)

    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Create a strike/maturity grid
x_grid = np.linspace(
    options["log_moneyness"].min(),
    options["log_moneyness"].max(),
    35,
)

T_grid = np.linspace(
    max(options["T"].min(), 7 / 365),
    options["T"].max(),
    30,
)

X, TT = np.meshgrid(x_grid, T_grid)
strikes = spot * np.exp(X)

# Obtain IV and Black-Scholes price at every grid point
iv = iv_surface(X, TT)
prices = call_price(spot, strikes, TT, r, iv)

# Calculate numerical price derivatives
dC_dT = np.gradient(prices, T_grid, axis=0)
dC_dK = np.gradient(prices, strikes[0], axis=1)
d2C_dK2 = np.gradient(dC_dK, strikes[0], axis=1)

# Apply the Dupire formula
numerator = dC_dT + r * strikes * dC_dK
denominator = 0.5 * strikes**2 * d2C_dK2

local_variance = numerator / denominator

# Replace invalid results with implied variance
valid = np.isfinite(local_variance) & (local_variance > 0)
local_variance = np.where(valid, local_variance, iv**2)

local_volatility = np.sqrt(
    np.clip(local_variance, 0.01**2, 2.0**2)
)

print("Local-volatility surface shape:", local_volatility.shape)
print("Approximate ATM local volatility:", local_volatility[:, 17])

Local-volatility surface shape: (30, 35)
Approximate ATM local volatility: [0.19296568 0.19925206 0.20101975 0.20284962 0.20352474 0.20359438
 0.20466042 0.20593286 0.20720988 0.20848874 0.20976753 0.21104485
 0.21126901 0.21046161 0.21050184 0.21133509 0.21216629 0.21299515
 0.21382145 0.21464498 0.21546561 0.2162832  0.21709766 0.2179089
 0.21871685 0.21952146 0.22032269 0.22112049 0.22191484 0.22323244]


In [4]:
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import norm
import numpy as np

#3. Using that local volatility in Black-Scholes to calculate the option’s delta.
# Interpolate the local-volatility grid
local_vol_surface = RegularGridInterpolator(
    (T_grid, x_grid),
    local_volatility,
    bounds_error=False,
    fill_value=None,
)

def dupire_delta(S, K, T, r):
    log_moneyness = np.log(K / S)

    # Look up local volatility
    sigma_local = float(
        local_vol_surface([[T, log_moneyness]])[0]
    )

    # Black-Scholes delta using local volatility
    d1 = (
        np.log(S / K)
        + (r + 0.5 * sigma_local**2) * T
    ) / (sigma_local * np.sqrt(T))

    return norm.cdf(d1)

# Example: ATM call with 30 days remaining
delta = dupire_delta(
    S=spot,
    K=spot,
    T=30 / 365,
    r=r,
)

print("Dupire delta:", delta)
print("Shares needed to hedge one short call:", delta * 100)

Dupire delta: 0.5347435425562717
Shares needed to hedge one short call: 53.47435425562716


In [5]:
#4. For one short call, buying approximately delta × 100 SPY shares.
# One short call contract represents 100 shares
option_quantity = -1
contract_multiplier = 100

# Offset the option position's negative delta
option_position_delta = option_quantity * delta * contract_multiplier
shares_to_buy = -option_position_delta

print("Call delta:", delta)
print("Shares of SPY to buy:", shares_to_buy)

Call delta: 0.5347435425562717
Shares of SPY to buy: 53.47435425562716


In [6]:
#5. Rebalancing the shares daily as the calculated delta changes.
current_shares = 0

for _, row in options.sort_values("date").iterrows():
    S = row["underlying_price"]
    K = row["strike"]
    T = row["T"]

    # Recalculate delta using today's information
    delta = dupire_delta(S, K, T, r)

    # Target shares for one short call
    target_shares = delta * 100

    # Shares that must be bought or sold today
    shares_to_trade = target_shares - current_shares

    print(
        row["date"],
        "Delta:", round(delta, 4),
        "Target:", round(target_shares, 2),
        "Trade:", round(shares_to_trade, 2),
    )

    current_shares = target_shares

2020-03-23 00:00:00 Delta: 0.9931 Target: 99.31 Trade: 99.31
2020-03-23 00:00:00 Delta: 0.9185 Target: 91.85 Trade: -7.46
2020-03-23 00:00:00 Delta: 0.8852 Target: 88.52 Trade: -3.33
2020-03-23 00:00:00 Delta: 0.8423 Target: 84.23 Trade: -4.29
2020-03-23 00:00:00 Delta: 0.7876 Target: 78.76 Trade: -5.47
2020-03-23 00:00:00 Delta: 0.7219 Target: 72.19 Trade: -6.57
2020-03-23 00:00:00 Delta: 0.6474 Target: 64.74 Trade: -7.46
2020-03-23 00:00:00 Delta: 0.5672 Target: 56.72 Trade: -8.01
2020-03-23 00:00:00 Delta: 0.4859 Target: 48.59 Trade: -8.13
2020-03-23 00:00:00 Delta: 0.4079 Target: 40.79 Trade: -7.8
2020-03-23 00:00:00 Delta: 0.337 Target: 33.7 Trade: -7.09
2020-03-23 00:00:00 Delta: 0.2756 Target: 27.56 Trade: -6.14
2020-03-23 00:00:00 Delta: 0.2246 Target: 22.46 Trade: -5.1
2020-03-23 00:00:00 Delta: 0.1839 Target: 18.39 Trade: -4.08
2020-03-23 00:00:00 Delta: 0.1608 Target: 16.08 Trade: -2.31
2020-03-23 00:00:00 Delta: 0.9419 Target: 94.19 Trade: 78.11
2020-03-23 00:00:00 Delta: 0